# 05 – Model Comparison

This notebook loads the saved weights for every model trained in this repository and produces a **side-by-side comparison** on the GTSRB test set:

| Model | Description |
|-------|-------------|
| Custom CNN | 3-block CNN trained from scratch |
| ResNet-50 | Transfer learning from ImageNet |
| MobileNetV2 | Lightweight transfer learning from ImageNet |

**Metrics reported:** Accuracy, Precision, Recall, F1-Score, Inference speed.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

sns.set_style('whitegrid')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT = Path('../data')
NUM_CLASSES = 43
BATCH_SIZE = 64

print(f'Device: {DEVICE}')

## 1. Load test set

In [ ]:
# Standard ImageNet normalisation (used by ResNet and MobileNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# We use 224×224 for all models here for a fair comparison;
# the CNN can handle any input size.
IMG_SIZE = 224

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_dataset = datasets.GTSRB(
    root=DATA_ROOT, split='test', download=True, transform=test_transform
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Test samples: {len(test_dataset):,}')

## 2. Re-create model architectures

In [ ]:
# ── Custom CNN ────────────────────────────────────────────────────────────────
class TrafficSignCNN(nn.Module):
    def __init__(self, num_classes: int = 43):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25),
        )
        # Adaptive pool so the classifier works regardless of input size
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512), nn.ReLU(True), nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))


# ── ResNet-50 ─────────────────────────────────────────────────────────────────
def build_resnet50(num_classes: int = 43) -> nn.Module:
    m = models.resnet50(weights=None)
    m.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(m.fc.in_features, num_classes))
    return m


# ── MobileNetV2 ───────────────────────────────────────────────────────────────
def build_mobilenetv2(num_classes: int = 43) -> nn.Module:
    m = models.mobilenet_v2(weights=None)
    in_features = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, num_classes))
    return m

## 3. Load saved weights

In [ ]:
def load_model(builder_or_instance, weights_path: Path, device):
    if callable(builder_or_instance):
        model = builder_or_instance()
    else:
        model = builder_or_instance
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model = model.to(device)
    model.eval()
    return model


WEIGHTS = {
    'Custom CNN':   DATA_ROOT / 'cnn_scratch_weights.pth',
    'ResNet-50':    DATA_ROOT / 'resnet50_weights.pth',
    'MobileNetV2':  DATA_ROOT / 'mobilenetv2_weights.pth',
}

BUILDERS = {
    'Custom CNN':  lambda: TrafficSignCNN(NUM_CLASSES),
    'ResNet-50':   lambda: build_resnet50(NUM_CLASSES),
    'MobileNetV2': lambda: build_mobilenetv2(NUM_CLASSES),
}

MODEL_NOTEBOOKS = {
    'Custom CNN':  '02',
    'ResNet-50':   '03',
    'MobileNetV2': '04',
}

models_dict = {}
for name, path in WEIGHTS.items():
    if path.exists():
        models_dict[name] = load_model(BUILDERS[name], path, DEVICE)
        print(f'✔  Loaded {name} from {path}')
    else:
        print(f'✘  Weights not found for {name}: {path} – run notebook {MODEL_NOTEBOOKS[name]} first')

## 4. Inference & metrics

In [ ]:
@torch.no_grad()
def run_inference(model, loader, device):
    all_preds, all_labels = [], []
    t0 = time.time()
    for images, labels in loader:
        preds = model(images.to(device)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
    elapsed = time.time() - t0
    return np.array(all_preds), np.array(all_labels), elapsed


results = {}
for name, model in models_dict.items():
    y_pred, y_true, elapsed = run_inference(model, test_loader, DEVICE)
    results[name] = {
        'y_pred': y_pred,
        'y_true': y_true,
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall':    recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1':        f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'inference_s': elapsed,
    }
    print(f'{name:15s} | acc={results[name]["accuracy"]:.4f} | '
          f'f1={results[name]["f1"]:.4f} | {elapsed:.1f}s')

## 5. Summary table

In [ ]:
summary = pd.DataFrame({
    name: {
        'Accuracy':       f"{r['accuracy']:.4f}",
        'Precision (w)':  f"{r['precision']:.4f}",
        'Recall (w)':     f"{r['recall']:.4f}",
        'F1 (weighted)':  f"{r['f1']:.4f}",
        'Inference (s)':  f"{r['inference_s']:.2f}",
    }
    for name, r in results.items()
}).T

display(summary)

## 6. Bar-chart comparison

In [ ]:
metrics = ['accuracy', 'precision', 'recall', 'f1']
labels  = [m.capitalize() for m in metrics]
model_names = list(results.keys())
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
for i, name in enumerate(model_names):
    values = [results[name][m] for m in metrics]
    ax.bar(x + i * width, values, width, label=name)

ax.set_xticks(x + width)
ax.set_xticklabels(labels, fontsize=12)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('GTSRB – Model comparison (weighted metrics)', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

## 7. Per-class F1 heatmap

In [ ]:
from sklearn.metrics import f1_score as f1_per_class

per_class_f1 = {}
for name, r in results.items():
    per_class_f1[name] = f1_per_class(
        r['y_true'], r['y_pred'], average=None, labels=list(range(43)), zero_division=0
    )

df_f1 = pd.DataFrame(per_class_f1, index=[f'Class {i}' for i in range(43)])

fig, ax = plt.subplots(figsize=(10, 16))
sns.heatmap(df_f1, ax=ax, cmap='YlGnBu', vmin=0, vmax=1,
            annot=True, fmt='.2f', linewidths=0.3, annot_kws={'size': 7})
ax.set_title('Per-class F1 score comparison', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Inference speed comparison

In [ ]:
names   = list(results.keys())
speeds  = [results[n]['inference_s'] for n in names]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(names, speeds, color=['steelblue', 'salmon', 'seagreen'])
ax.bar_label(bars, fmt='%.2f s', padding=4)
ax.set_xlabel('Total inference time (seconds)')
ax.set_title(f'Inference time on test set ({len(test_dataset):,} images)')
plt.tight_layout()
plt.show()